<a href="https://colab.research.google.com/github/soot-bit/pinnslicer/blob/main/notebooks/01_pinn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhotonOrbitSolver Training
> African Institute of Mathematical Sciences (AIMS), Cape Town, South Africa<br>
> Created: March 2025 Claire David, Tlotlo M. Oepeng, Harrison B. Prosper<br>
> Updated: Tue Sep 1 2026 HBP : use torch.jit.script to save both the model<br>
>          definition and the fitted parameters to a file along with a few<br>
>          non-trainable parameters.<br>
>          Thu Sep 3 2026 HBP: rollback changes. 1) torch.jit.script requires too
>          many changes and 2) storing dPhi and using recursive saving of parameters
>          starting with the Solution object changes the names of the parameters!<br>
> Updated: Fri Sep 4 2026: HBP: Write a more efficient algorithm for pinn_solver.

## Introduction
This notebook trains a Physics-Informed Neural Network (PINN) [1, 2] to solve the following nonlinear ordinary differential equation (ODE):

\begin{align}
  \ddot{u}  \: + \: u - \tfrac{3}{2}u^2  & = \: 0,
\end{align}
where
\begin{align}
    \ddot{u} & = \frac{d^2u}{d\phi^2},\\
    u & = \frac{r_s}{r}, \: \text{ and} \\
    r_s & = \frac{2 G M}{c^2},
\end{align}
is the Schwarzschild radius; $G$ is Newton's gravitational constant, $M$ is  the mass of a spherically symmetric body, and $c$ is the speed of light in vacuum.
 If $C$ is the proper circumference of a circle centered at the center of mass, then
the radial coordinate, $r$, is defined by $r \equiv C \, / \, (2\pi)$, which differs from the proper radial distance. By definition, the proper circumference is a circle along which the time is the same at every point.

The ODE describes the orbit of photons in the Schwarzschild spacetime about the spherically symmetruc body. We shall refer to this equation as the **photon orbit equation**.
The angle $\phi$ is the azimuthal angle in a spherical polar coordinate system, $(r, \theta, \phi)$. Here $\theta$ is set to $\pi \, / \, 2$ without loss of generality. The initial conditions are
\begin{align}
u\,(0) &= u_0 \\
\overset{\textstyle\cdot}{u}\,(0) &= v_0.
\end{align}

### Approach
The ODE is solved using a PINN following the approach in [3]. The neural network is described by the function $g_\beta(\phi, u_0, v_0),$ where $\beta$ are the network's trainable parameters.  We use the following *ansatz* from the theory of connections (ToC) [4] that incorporates the initial conditions explicitly:

\begin{align}
    u(\phi, u_0, v_0)  &= u_0 + g_\beta(\phi, u_0, v_0) - g_\beta(0, u_0, v_0) + \phi \left[ v_0 - \dot{g}_\beta(0, u_0, v_0) \right], \\[1ex]
    \dot{u}(\phi, u_0, v_0) &= v_0 + \dot{g}_\beta(\phi, u_0, v_0) - \dot{g}_\beta(0, u_0, v_0),
\end{align}

### References
[1] B. Moseley, [Deep Learning in Scientific Computing (2023)](https://camlab.ethz.ch/teaching/deep-learning-in-scientific-computing-2023.html), ETH Zürich, Computational and Applied Mathematics Laboratory (CAMLab)  
[2] S. Cuomo *et al*., *Scientific Machine Learning through Physics-Informed Neural Networks: Where we are and What's next*, [arXiv:2201.05624](https://doi.org/10.48550/arXiv.2201.05624)  
[3] Aditi S. Krishnapriyan, Amir Gholami, Shandian Zhe, Robert M. Kirby, Michael W. Mahoney, *Characterizing possible failure modes in physics-informed neural networks*, NIPS'21: Proceedings of the 35th International Conference on Neural Information Processing Systems; [arXiv:2109.01050](https://arxiv.org/abs/2109.01050)  
[4] D. Mortari, *The Theory of Connections: Connecting Points*, Mathematics, vol. 5, no. 57, 2017.


## Local installation `pinnslicer`
  ```bash
      git clone https://github.com/soot-bit/pinnslicer
      cd pinnslicer
      pip install -e .
  ```
## To run on Google Colab
Go to: https://colab.research.google.com

# Imports

In [1]:
try:
    import google.colab
    google.colab.drive.mount('/content/gdrive')

    !git clone https://github.com/soot-bit/pinnslicer.git > /dev/null 2>&1
    !cd /content/pinnslicer; pip install .

    # give path to your folder on Google Drive
    DIRPATH  = '/content/gdrive/MyDrive/PINN'
    IN_COLAB = True

except ImportError:

    DIRPATH  = ''
    IN_COLAB = False
# -----------------------------------------------------------------------
import os
import sys
import importlib
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import MultiStepLR

import warnings
warnings.filterwarnings('ignore', message='.*no current CUDA context.*')

# PINN library
import pinnslicer.nn as mlp
import pinnslicer.utils.data as dat
from pinnslicer.utils.seeding import set_seed

## Setup

In [3]:
# -----------------------------
# Hardware
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# -----------------------------
# Configuration
# -----------------------------

# log-file folder: [<dirpath>/]runs/<dirname>
dirpath = DIRPATH

# Run identifier. 
dirname = 'pinn-26-09-12'
name    = 'pinn-26-09-12'

# choose whether to create or load a configuration file
load_existing_config = False

if load_existing_config:
    # specify name of config file
    cfg_filename = f'runs/{dirname}/{name}_config.yaml'

    config = mlp.Config(cfg_filename)
else:
    # create new configuration. If dirname is omitted then
    # a time-stamped folder is created.
    config = mlp.Config(name, dirname=dirname, dirpath=dirpath)

    # Phi segement size
    config('dPhi', 0.1)

    # Bounds
    #                       Phi     u0    v0
    config('lower_bounds', [0.0,  0.10, -1.0])
    config('upper_bounds', [config('dPhi'), 0.99,  1.0])

    # training configuration
    # -----------------------------------------
    config('dataset_size_exponent', 16)
    config('train_size', 2**config('dataset_size_exponent'))
    config('val_size',     5000)  # validation sample size
    config('batch_size',   2048)  #
    config('monitor_step', 2000)  # monitor training every n (=10) iterations
    config('delete', True)        # delete losses file before training, if True

    # optimizer / scheduler configuration
    # -----------------------------------------
    # a step comprises a given number of iterations
    config('num_steps', 5)        # number of training steps
    config('num_iters_per_step', 200_000)
    config('base_lr', 3e-3)       # initial learning rate
    config('gamma', 0.24)         # learning rate scale factor

    config('val_cost_drop_threshold', 0.005)

# -----------------------------
# Reproducibility
# -----------------------------
# Both the seed and the network architecture belong in the configuration
# file: the seed so the run can be repeated, the architecture so the saved
# weights can be reloaded into a network of the right shape without anyone
# having to remember what the defaults were on the day.
#
# Reading a key with no value returns it; reading a key that does not exist
# raises KeyError, which is how a new configuration is told from a loaded one.
try:
    seed = config('seed')
except KeyError:
    seed = config('seed', 1234)

try:
    arch = config('arch')
except KeyError:
    arch = config('arch', {'n_inputs': 3, 'n_hidden': 3, 'n_width': 75})

# Seeds python, numpy and torch. The Sobol scrambler does not read the global
# numpy seed, so it is seeded explicitly where the sample is drawn.
set_seed(seed)

if not load_existing_config:
    print(f'\nSave configuration to file {config.cfg_filename}\n')
    config.save()

# Total number of iterations
config('num_iterations', config('num_iters_per_step') * config('num_steps'))

print(config)

# -----------------------------
# Live Plotting Options
# -----------------------------
live_display  = False    # to plot cost evolution during training


Using device: cpu

  Seeded random, numpy and torch with seed = 1234
  (Sobol sampling is seeded separately: SobolSample(seed=1234))

Save configuration to file runs/pinn-26-09-12/pinn-26-09-12_config.yaml

name: pinn-26-09-12
file:
  losses: runs/pinn-26-09-12/pinn-26-09-12_losses.csv
  params: runs/pinn-26-09-12/pinn-26-09-12_params.pth
  script: runs/pinn-26-09-12/pinn-26-09-12_script.pth
  init_params: runs/pinn-26-09-12/pinn-26-09-12_init_params.pth
  plots: runs/pinn-26-09-12/pinn-26-09-12_plots.png
dPhi: 0.1
lower_bounds:
- 0.0
- 0.1
- -1.0
upper_bounds:
- 0.1
- 0.99
- 1.0
dataset_size_exponent: 16
train_size: 65536
val_size: 5000
batch_size: 2048
monitor_step: 2000
delete: true
num_steps: 5
num_iters_per_step: 200000
base_lr: 0.003
gamma: 0.24
val_cost_drop_threshold: 0.005
seed: 1234
arch:
  n_inputs: 3
  n_hidden: 3
  n_width: 75
num_iterations: 1000000



# Datasets

In [4]:
# Raw training data
# Note: each sample gets its own seed, derived from the run seed, so that the
# training points, the validation points and the training subset are drawn
# independently but reproducibly.
data_train = dat.SobolSample(
    config('lower_bounds'),
    config('upper_bounds'),
    num_points_exp=config('dataset_size_exponent'),
    seed=config('seed')
)

# Validation data
data_val = dat.UniformSample(
    config('lower_bounds'),
    config('upper_bounds'),
    num_points=config('val_size'),
    seed=config('seed') + 1
)

# -----------------------------
# Tensor datasets
# -----------------------------
# Dataset for training: full tensorized subset from the raw sampling
print('\n===> Creating: train_dataset')
train_dataset = dat.Dataset(
    data_train,
    start=0, end=config('train_size'),
    verbose=1,
    device=device
)

# Training subset of same size as validation dataset
print('\n===> Creating: train_valsize_dataset')
train_valsize_dataset = dat.Dataset(
    data_train,
    start=0, end=config('train_size'),
    random_sample_size=config('val_size'),
    seed=config('seed') + 2,
    verbose=1,
    device=device
)

# Dataset for validation: tensorized points from the raw uniform sampling
print('\n===> Creating: val_dataset')
val_dataset = dat.Dataset(
    data_val,
    start=0, end=config('val_size'),
    verbose=1,
    device=device
)


  SobolSample
  65536 Sobol points created (seed: 1234).
  UniformSample
  5000 uniformly sampled points created (seed: 1235).

===> Creating: train_dataset
  Type               : Dataset
  Shape of phi_vals  : torch.Size([65536, 1])
  Shape of init_conds: torch.Size([65536, 2])

===> Creating: train_valsize_dataset
  Type               : Dataset
  Shape of phi_vals  : torch.Size([5000, 1])
  Shape of init_conds: torch.Size([5000, 2])

===> Creating: val_dataset
  Type               : Dataset
  Shape of phi_vals  : torch.Size([5000, 1])
  Shape of init_conds: torch.Size([5000, 2])


# DataLoaders

In [5]:
# Loader for main training batches
print('\n===> Creating: train_loader')
train_loader = dat.DataLoader(train_dataset,
                              batch_size=config('batch_size'),
                              num_iterations=config('num_iterations'),
                              shuffle=True,
                              seed=config('seed') + 3
)

# Loader for evaluating validation cost (single batch of val_size)
print('\n===> Creating: val_loader')
val_loader = dat.DataLoader(val_dataset,
                            batch_size=config('val_size')
)

# Loader for evaluating training cost with val-sized batch
print('\n===> Creating: train_valsize_loader')
train_valsize_loader = dat.DataLoader(train_valsize_dataset,
                                      batch_size=config('val_size')
)



===> Creating: train_loader
DataLoader
  Number of iterations has been specified
  maxiter:         1000000
  batch_size:         2048
  shuffle_step:         32


===> Creating: val_loader
DataLoader
  maxiter:               1
  batch_size:         5000
  shuffle_step:          1


===> Creating: train_valsize_loader
DataLoader
  maxiter:               1
  batch_size:         5000
  shuffle_step:          1



## Model, Solution, Objective

In [6]:
# Model Instantiation
print('\n===> Creating model...\n')

# architecture read from the configuration, so that it is recorded with the run
fcnn_model = mlp.FCNN(**config('arch')).to(device)

pinn_soln  = mlp.Solution(fcnn_model).to(device)

pinn_obj = mlp.Objective(pinn_soln).to(device)

print(pinn_soln)
print(f'Number of parameters: {mlp.count_trainable_parameters(pinn_soln)}')



===> Creating model...

Solution(
  (g): FCNN(
    (model): ModuleList(
      (0): Linear(in_features=3, out_features=75, bias=True)
      (1): Sin()
      (2): Linear(in_features=75, out_features=75, bias=True)
      (3): Sin()
      (4): Linear(in_features=75, out_features=75, bias=True)
      (5): Sin()
    )
    (output_layer): Linear(in_features=75, out_features=1, bias=True)
  )
)
Number of parameters: 11776


# Scheduler
Using a multistep scheduler parametrized with $\gamma$ (initially 0.24).


In [7]:
# Instantiate optimizer with base learning rate
print("\n===> Creating optimizer...\n")
print(f"    Base learning rate: {config('base_lr'):10.1e}")
optimizer = torch.optim.Adam(pinn_soln.parameters(), lr=config('base_lr'))

# Learning rate milestones (after n_step iterations)
n_milestones = config('num_steps') - 1
print(f'    Number of milestones: {n_milestones:5d}\n')
milestones = [n * config('num_iters_per_step') for n in range(config('num_steps'))]

print("\n===> Creating scheduler...\n")
# Drop first entry of milestones list because it contains the base LR
scheduler = MultiStepLR(optimizer, milestones=milestones[1:], gamma=config('gamma'))

mlp.print_milestones_and_lrs(config('base_lr'),
                             config('num_steps'),
                             milestones,
                             config('gamma'),
                             n_max_iterations=config('num_iterations'))


===> Creating optimizer...

    Base learning rate:    3.0e-03
    Number of milestones:     4


===> Creating scheduler...

Step | Milestone | LR
-----------------------------
   0 |         0 | 3.0e-03   
-----------------------------
   1 |    200000 | 7.2e-04   
   2 |    400000 | 1.7e-04   
   3 |    600000 | 4.1e-05   
   4 |    800000 | 1.0e-05   

Total number of iterations:    1000000



# Training Loop

In [ ]:
mlp.train_pinn(
    train_loader, val_loader, train_valsize_loader,
    optimizer, scheduler, pinn_obj,
    display_costs=live_display,
    model_filename=config('file/params'),
    log_filename=config('file/losses'),
    plot_filename=config('file/plots'),
    monitor_every_n_iterations=config('monitor_step'),
    drop_threshold=config('val_cost_drop_threshold')
)